# Ground Truth : démarrage sur Colab (GPU T4)

1. Exécution > Modifier le type d'exécution > **GPU T4**.
2. Exécution > **Tout exécuter**. Compter ~20 min la première fois (corpus, puis 4,7 Go de poids du générateur et 8 Go pour le juge).

Si Colab demande l'accès au secret HF_TOKEN, tu peux cliquer sur Annuler (ou ajouter ton jeton Hugging Face dans Secrets pour télécharger plus vite).

Le notebook télécharge le corpus, lance la baseline sur 20 questions de validation et les note avec le juge officiel.

In [ ]:
!git clone -q https://github.com/Mathos34/ground-truth-kit.git
%cd ground-truth-kit
!pip install -q -r requirements.txt

In [ ]:
!python corpus/download_corpus.py

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Le générateur officiel

Le test se lance dans un **processus séparé** (`!python -m groundtruth.generator`) : à la fin du processus, la VRAM est libérée. Ne garde pas le générateur chargé dans le noyau du notebook pendant qu'un script le recharge : le modèle occupe environ 7,3 Go, deux exemplaires ne tiennent pas sur un T4, et `del` + `torch.cuda.empty_cache()` ne les libèrent pas vraiment (d'où l'erreur « CUDA out of memory » à l'étape suivante).

Dans ton propre code, tu utiliseras la classe ainsi :

```python
from groundtruth.generator import OfficialGenerator

gen = OfficialGenerator(backend="hf")
gen.start_question("q1")
msgs = [{"role": "system", "content": "Réponds en une phrase, en français."},
        {"role": "user", "content": "Quelle est la capitale de la France ?"}]
print(gen.count_tokens(msgs), "tokens")
print(gen.chat(msgs, max_new_tokens=50))
```

In [ ]:
!python -m groundtruth.generator

## Baseline BM25 sur 20 questions de validation

In [ ]:
!python -m baseline.pipeline index --pdf-dir corpus/pdf --index-dir index/
import json
v = json.load(open('data/validation_v1.json'))
v['questions'] = v['questions'][:20]
json.dump(v, open('runs_val20.json', 'w'), ensure_ascii=False)
!python -m baseline.pipeline answer --index-dir index/ --questions runs_val20.json --out runs/val20.json --backend hf

In [ ]:
!python -m groundtruth.evaluate --pred runs/val20.json --gold runs_val20.json --judge hf

## Palier 3 : mise à jour du corpus

`update` ajoute les 20 documents v2 à l'index existant, sans tout réindexer.

In [ ]:
!python -m baseline.pipeline update --index-dir index/ --pdf-dir corpus/pdf_v2